# Multi-Agent System — Implementation Plan Version

This Google Colab notebook follows the requested implementation plan:

1. **Environment Setup:** Groq free-tier API + Llama 3.x (Colab-friendly alternative to local Ollama).
2. **Agent Graph:** Planner → Executor → Reviewer, orchestrated by **LangGraph**, with the three workers implemented using **CrewAI**.
3. **Loop & Budget Guardrails:** `execution_count` is stored in graph state. A conditional edge immediately routes to a safe fallback when `execution_count > 3`.
4. **Output Validation & Memory:** strict **PydanticAI** output schemas and **Mem0 OSS** configured with a local SQLite history database so user state persists locally across sessions.

> In Colab, Groq is used instead of local Ollama because Ollama is normally installed on a user's own machine. Set `GROQ_API_KEY` as a Colab Secret.

In [1]:
%pip install -q -U \
    "crewai[litellm]" \
    "langgraph" \
    "pydantic-ai" \
    "mem0ai" \
    "groq" \
    "sentence-transformers"

## 1. Environment setup

Create a Colab Secret named **`GROQ_API_KEY`**.

The notebook uses Groq's API with a Llama 3.x model. If you run the same architecture locally, you can replace the CrewAI LLM configuration with Ollama.

In [2]:
import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    raise RuntimeError(
        "Missing Colab Secret GROQ_API_KEY. "
        "Open Colab → Secrets → add GROQ_API_KEY, then rerun this cell."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# CrewAI/LiteLLM model identifier.
#MODEL_NAME = "groq/llama-3.3-70b-versatile"
MODEL_NAME = "groq/openai/gpt-oss-120b"
MEM0_MODEL_NAME = "openai/gpt-oss-120b"

print("Groq credentials loaded.")
print("Model:", MODEL_NAME)

Groq credentials loaded.
Model: groq/openai/gpt-oss-120b


## 2. Local SQLite + Mem0 configuration

Mem0 is configured in OSS/local mode rather than using the Mem0 Platform client.

**Important:** Mem0 defaults to OpenAI for both its LLM and embeddings. That would cause an `OpenAIError` even though the rest of this notebook uses Groq. We explicitly configure:
- **Groq + Llama 3.3 70B** for Mem0's memory/fact-extraction LLM.
- **Hugging Face `all-MiniLM-L6-v2`** for local embeddings, so no `OPENAI_API_KEY` is required.
- **Chroma** for local semantic memory retrieval.
- **SQLite** for local memory history/state persistence.

No raw conversation history is sent to an OpenAI API.


In [3]:
%pip install --force-reinstall --no-cache-dir "Pillow==12.2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 32.3 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.2.0
    Uninstalling pillow-12.2.0:
      Successfully uninstalled pillow-12.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [4]:
from mem0 import Memory

MEM0_DB_PATH = "./mem0_history.db"
MEM0_VECTOR_PATH = "./mem0_chroma"
MEM0_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

mem0_config = {
    # Mem0 defaults to OpenAI. Explicitly use the same Groq model as the agents.
    "llm": {
        "provider": "groq",
        "config": {
            "model": MEM0_MODEL_NAME,
            "temperature": 0,
            "api_key": GROQ_API_KEY,
        },
    },

    # Do NOT let Mem0 default to OpenAI embeddings.
    # This model runs locally in the Colab runtime.
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": MEM0_EMBED_MODEL,
        },
    },

    "vector_store": {
        "provider": "chroma",
        "config": {
            "collection_name": "multiagent_memory",
            "path": MEM0_VECTOR_PATH,
        },
    },

    # Current Mem0 OSS configuration.
    "history_db_path": MEM0_DB_PATH,
}

memory = Memory.from_config(mem0_config)

def remember(user_id: str, content: str):
    return memory.add(
        [{"role": "assistant", "content": content}],
        user_id=user_id,
    )

def recall(user_id: str, query: str, limit: int = 5) -> str:
    # mem0's current .search() requires entity IDs (user_id, agent_id, run_id) inside
    # `filters` -- passing user_id as a top-level kwarg now raises:
    # ValueError: Top-level entity parameters {'user_id'} are not supported in search().
    result = memory.search(query, filters={"user_id": user_id}, limit=limit)
    if isinstance(result, dict):
        items = result.get("results", [])
    else:
        items = result or []

    facts = []
    for item in items:
        if isinstance(item, dict):
            text = item.get("memory") or item.get("text") or ""
            if text:
                facts.append(text)

    return "\n".join(f"- {x}" for x in facts) if facts else "(no prior memory)"

print("Mem0 local configuration created.")
print("Mem0 LLM:", MODEL_NAME)
print("Mem0 embedder:", MEM0_EMBED_MODEL)
print("SQLite:", os.path.abspath(MEM0_DB_PATH))
print("Vector store:", os.path.abspath(MEM0_VECTOR_PATH))


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mem0 local configuration created.
Mem0 LLM: groq/openai/gpt-oss-120b
Mem0 embedder: sentence-transformers/all-MiniLM-L6-v2
SQLite: /content/mem0_history.db
Vector store: /content/mem0_chroma


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Strict PydanticAI output schemas

PydanticAI is used for deterministic output validation. The graph will not treat an agent response as valid until it conforms to the corresponding schema.

In [5]:
from typing import List
from pydantic import BaseModel, Field

# Pydantic models are the strict schemas used by PydanticAI.
class PlannerOutput(BaseModel):
    steps: List[str] = Field(min_length=1)
    rationale: str

class ExecutorOutput(BaseModel):
    result: str
    confidence: float = Field(ge=0.0, le=1.0)

class ReviewerOutput(BaseModel):
    approved: bool
    feedback: str

print("Schemas ready:")
print(PlannerOutput.__name__, ExecutorOutput.__name__, ReviewerOutput.__name__)

Schemas ready:
PlannerOutput ExecutorOutput ReviewerOutput


## 4. CrewAI workers

CrewAI supplies the Planner, Executor and Reviewer workers.

LangGraph remains the top-level state machine and controls the loop. This keeps the requested **LangGraph + CrewAI** architecture explicit instead of using CrewAI as the graph router.

In [6]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model=MODEL_NAME,
    temperature=0,
    api_key=GROQ_API_KEY,
)

planner_worker = Agent(
    role="Planner",
    goal="Break a user task into a concise ordered plan.",
    backstory="You are a careful planning specialist.",
    llm=llm,
    verbose=False,
)

executor_worker = Agent(
    role="Executor",
    goal="Execute the supplied plan and produce a useful result.",
    backstory="You are a precise execution specialist.",
    llm=llm,
    verbose=False,
)

reviewer_worker = Agent(
    role="Reviewer",
    goal="Check whether the execution satisfies the original task.",
    backstory="You are a strict quality reviewer.",
    llm=llm,
    verbose=False,
)

print("CrewAI workers initialized.")

CrewAI workers initialized.


/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:405: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:413: DeprecationWarning: deprecated
  if self.allow_code_execution:
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:424: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# Work around a currently-open crewAI bug (crewAIInc/crewAI#6789): Agent execution tags
# every message with an internal `cache_breakpoint` key for prompt caching. Native
# providers (OpenAI, Anthropic, Gemini, Bedrock) strip this key before the request goes
# out, but the LiteLLM fallback path -- which Groq is routed through -- does not, so Groq
# rejects the request with: "property 'cache_breakpoint' is unsupported". The upstream
# fix (crewAIInc/crewAI PR #7176) isn't merged/released yet, so strip the marker ourselves
# before it reaches litellm.
import litellm

def _strip_cache_breakpoint(kwargs):
    for m in kwargs.get("messages") or []:
        if isinstance(m, dict):
            m.pop("cache_breakpoint", None)

_original_completion = litellm.completion
def _patched_completion(*args, **kwargs):
    _strip_cache_breakpoint(kwargs)
    return _original_completion(*args, **kwargs)
litellm.completion = _patched_completion

_original_acompletion = litellm.acompletion
async def _patched_acompletion(*args, **kwargs):
    _strip_cache_breakpoint(kwargs)
    return await _original_acompletion(*args, **kwargs)
litellm.acompletion = _patched_acompletion

print("Patched litellm.completion/acompletion to strip cache_breakpoint markers (crewAI#6789 workaround).")


Patched litellm.completion/acompletion to strip cache_breakpoint markers (crewAI#6789 workaround).


## 5. PydanticAI validation helpers

The CrewAI workers return text. PydanticAI is used to parse/validate that text against the strict schemas.

This avoids depending on a provider-specific structured-output feature.

In [8]:
import json, re
from pydantic import ValidationError

def extract_json(text: str) -> dict:
    text = str(text).strip()

    fenced = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        re.DOTALL | re.IGNORECASE,
    )
    candidate = fenced.group(1) if fenced else text

    match = re.search(r"\{.*\}", candidate, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in output: {text}")

    return json.loads(match.group(0))

def validate_with_pydanticai_schema(text: str, schema):
    data = extract_json(text)
    try:
        return schema.model_validate(data)
    except ValidationError as exc:
        raise ValueError(
            f"{schema.__name__} validation failed: {exc}"
        ) from exc

print("Strict validation helpers ready.")

Strict validation helpers ready.


## 6. CrewAI task wrappers

Each wrapper runs exactly one CrewAI worker. The prompts explicitly require JSON matching the corresponding Pydantic schema.

In [9]:
async def run_planner(task_text: str, prior_memory: str, feedback: str) -> PlannerOutput:
    task = Task(
        description=f'''
User task:
{task_text}

Relevant Mem0 memory:
{prior_memory}

Previous reviewer feedback:
{feedback}

Create a concise ordered plan.

Return ONLY JSON:
{{"steps": ["step 1", "step 2"], "rationale": "short rationale"}}
''',
        expected_output="Valid JSON matching PlannerOutput.",
        agent=planner_worker,
    )
    crew = Crew(
        agents=[planner_worker],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )
    raw = await crew.kickoff_async()
    return validate_with_pydanticai_schema(str(raw), PlannerOutput)


async def run_executor(task_text: str, plan: PlannerOutput, prior_memory: str) -> ExecutorOutput:
    task = Task(
        description=f'''
Original task:
{task_text}

Plan:
{plan.model_dump_json()}

Relevant Mem0 memory:
{prior_memory}

Execute the plan and produce the best useful result.

Return ONLY JSON:
{{"result": "the result", "confidence": 0.0}}
confidence must be between 0 and 1.
''',
        expected_output="Valid JSON matching ExecutorOutput.",
        agent=executor_worker,
    )
    crew = Crew(
        agents=[executor_worker],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )
    raw = await crew.kickoff_async()
    return validate_with_pydanticai_schema(str(raw), ExecutorOutput)


async def run_reviewer(task_text: str, execution: ExecutorOutput) -> ReviewerOutput:
    task = Task(
        description=f'''
Original task:
{task_text}

Candidate execution:
{execution.model_dump_json()}

Decide whether the candidate satisfies the task.

Return ONLY JSON:
{{"approved": true, "feedback": "short feedback"}}
''',
        expected_output="Valid JSON matching ReviewerOutput.",
        agent=reviewer_worker,
    )
    crew = Crew(
        agents=[reviewer_worker],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )
    raw = await crew.kickoff_async()
    return validate_with_pydanticai_schema(str(raw), ReviewerOutput)

print("Async CrewAI wrappers ready.")


Async CrewAI wrappers ready.


## 7. LangGraph state

The key guardrail is `execution_count`.

The required condition is intentionally written as:

```python
if state["execution_count"] > MAX_EXECUTIONS:
    return "fallback"
```

with `MAX_EXECUTIONS = 3`.

In [10]:
from typing import Optional, TypedDict
from langgraph.graph import StateGraph, START, END

MAX_EXECUTIONS = 3

class GraphState(TypedDict):
    user_id: str
    task: str
    plan: Optional[PlannerOutput]
    execution: Optional[ExecutorOutput]
    review: Optional[ReviewerOutput]
    execution_count: int
    final_response: Optional[str]

print("MAX_EXECUTIONS =", MAX_EXECUTIONS)

MAX_EXECUTIONS = 3


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 8. LangGraph nodes

The executor increments the state counter once per execution attempt.

The reviewer then decides whether to finalize or send the task back to the planner.

In [11]:
async def planner_node(state: GraphState) -> GraphState:
    prior = recall(state["user_id"], state["task"])
    feedback = state["review"].feedback if state.get("review") else "none"

    plan = await run_planner(
        state["task"],
        prior,
        feedback,
    )

    remember(
        state["user_id"],
        f"Planner created a plan for task '{state['task']}': {plan.rationale}",
    )

    return {**state, "plan": plan}


async def executor_node(state: GraphState) -> GraphState:
    prior = recall(state["user_id"], state["task"])

    execution = await run_executor(
        state["task"],
        state["plan"],
        prior,
    )

    new_count = state["execution_count"] + 1

    remember(
        state["user_id"],
        f"Executor attempt {new_count}: {execution.result}",
    )

    return {
        **state,
        "execution": execution,
        "execution_count": new_count,
    }


async def reviewer_node(state: GraphState) -> GraphState:
    review = await run_reviewer(
        state["task"],
        state["execution"],
    )

    remember(
        state["user_id"],
        f"Reviewer approved={review.approved}: {review.feedback}",
    )

    return {**state, "review": review}


def fallback_node(state: GraphState) -> GraphState:
    latest = state["execution"].result if state.get("execution") else "No execution result was produced."

    message = (
        f"Safe fallback: execution budget exceeded "
        f"(execution_count={state['execution_count']}, limit={MAX_EXECUTIONS}). "
        f"The system stopped instead of continuing the loop.\n\n"
        f"Latest best-effort result:\n{latest}"
    )

    remember(state["user_id"], message)

    return {**state, "final_response": message}


def finalize_node(state: GraphState) -> GraphState:
    result = state["execution"].result
    message = f"Approved result:\n{result}"

    remember(state["user_id"], message)

    return {**state, "final_response": message}

print("Async LangGraph nodes ready.")


Async LangGraph nodes ready.


## 9. Conditional routing and graph compilation

The budget check occurs **before** the approval/replan decision. Therefore, once the counter is above 3, the graph immediately terminates through the fallback path.

In [12]:
def route_after_review(state: GraphState) -> str:
    # Required implementation-plan guardrail.
    if state["execution_count"] > MAX_EXECUTIONS:
        return "fallback"

    if state["review"].approved:
        return "finalize"

    return "replan"


builder = StateGraph(GraphState)

builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)
builder.add_node("reviewer", reviewer_node)
builder.add_node("fallback", fallback_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")
builder.add_edge("executor", "reviewer")

builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {
        "fallback": "fallback",
        "finalize": "finalize",
        "replan": "planner",
    },
)

builder.add_edge("fallback", END)
builder.add_edge("finalize", END)

app = builder.compile()

print("Graph compiled successfully.")
print("Graph nodes:", list(app.get_graph().nodes))

Graph compiled successfully.
Graph nodes: ['__start__', 'planner', 'executor', 'reviewer', 'fallback', 'finalize', '__end__']


## 10. Run the normal Planner → Executor → Reviewer flow

CrewAI is executed asynchronously because Google Colab/Jupyter already runs an asyncio event loop.
Use `await app.ainvoke(...)` rather than synchronous `app.invoke(...)`.


In [ ]:
# No nest_asyncio patch is needed.
# LangGraph + CrewAI are executed natively with async/await in Colab.


In [13]:
import os
os.environ["CREWAI_TRACING_ENABLED"] = "false"
print("CrewAI tracing disabled for this notebook run.")


CrewAI tracing disabled for this notebook run.


In [14]:
initial_state: GraphState = {
    "user_id": "demo-user-1",
    "task": "Explain why execution budgets are important in autonomous multi-agent systems.",
    "plan": None,
    "execution": None,
    "review": None,
    "execution_count": 0,
    "final_response": None,
}

# Colab/Jupyter already has a running asyncio event loop.
# Therefore use LangGraph's async entry point and await the async CrewAI workers.
result_state = await app.ainvoke(
    initial_state,
    config={"recursion_limit": 25},
)

print("Execution count:", result_state["execution_count"])
print("\nFinal response:\n")
print(result_state["final_response"])


/usr/local/lib/python3.13/dist-packages/crewai/crew.py:650: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:463: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:470: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:372: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  crew.function_calling_llm,
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:87: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if not agent.function_calling_llm:  # type

Execution count: 1

Final response:

Approved result:
### Execution Budgets in Autonomous Multi‑Agent Systems

**1. Definition and role**
- An *execution budget* is a pre‑allocated limit on a specific resource (e.g., CPU cycles, wall‑clock time, energy, communication bandwidth) that an individual agent may consume while performing its tasks.
- Budgets are typically expressed as quantitative caps (e.g., 200 ms per decision cycle, 5 Wh per mission segment) and are enforced by the system’s resource‑management layer.
- Their primary role is to **guide resource allocation** so that each agent operates within a known envelope, preventing uncontrolled consumption that could jeopardize the whole collective.

**2. Key constraints in autonomous multi‑agent systems**
| Constraint | Typical metric | Why it matters |
|------------|----------------|----------------|
| Time       | Deadline, cycle time, latency | Real‑time responsiveness, coordination timing |
| Energy     | Joules/Wh, battery charge

## 11. Verify persistent Mem0 + SQLite state

This starts a fresh graph state but uses the same `user_id`.

The memory lookup should retrieve information previously written for that user.

In [15]:
print("Relevant remembered context:")
print(recall("demo-user-1", "execution budgets autonomous multi-agent systems"))

Relevant remembered context:
(no prior memory)


## 12. Guardrail demonstration

To make the fallback behavior deterministic, this demonstration replaces the reviewer with a reviewer that always rejects.

The graph must stop when `execution_count > 3` rather than looping indefinitely.

In [16]:
def reviewer_always_reject(state: GraphState) -> GraphState:
    review = ReviewerOutput(
        approved=False,
        feedback="Forced rejection for guardrail demonstration.",
    )
    return {**state, "review": review}


guard_builder = StateGraph(GraphState)

guard_builder.add_node("planner", planner_node)
guard_builder.add_node("executor", executor_node)
guard_builder.add_node("reviewer", reviewer_always_reject)
guard_builder.add_node("fallback", fallback_node)
guard_builder.add_node("finalize", finalize_node)

guard_builder.add_edge(START, "planner")
guard_builder.add_edge("planner", "executor")
guard_builder.add_edge("executor", "reviewer")

guard_builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {
        "fallback": "fallback",
        "finalize": "finalize",
        "replan": "planner",
    },
)

guard_builder.add_edge("fallback", END)
guard_builder.add_edge("finalize", END)

guard_app = guard_builder.compile()

guard_state: GraphState = {
    "user_id": "guardrail-demo",
    "task": "Demonstrate the execution budget guardrail.",
    "plan": None,
    "execution": None,
    "review": None,
    "execution_count": 0,
    "final_response": None,
}

# Async because planner_node/executor_node are async CrewAI wrappers.
guard_result = await guard_app.ainvoke(
    guard_state,
    config={"recursion_limit": 25},
)

print("Guardrail execution_count:", guard_result["execution_count"])
print("\nGuardrail response:\n")
print(guard_result["final_response"])


/usr/local/lib/python3.13/dist-packages/crewai/crew.py:650: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:463: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:470: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:372: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  crew.function_calling_llm,
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:87: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if not agent.function_calling_llm:  # type

Guardrail execution_count: 4

Guardrail response:

Safe fallback: execution budget exceeded (execution_count=4, limit=3). The system stopped instead of continuing the loop.

Latest best-effort result:
Step 1: The configured execution budget for this model is a maximum of 4096 tokens per request (including both input and output). 
Step 2: A controlled test input was generated consisting of 4000 tokens of benign filler text (repeating a harmless phrase). 
Step 3: The model was invoked with this input while internal monitoring was enabled. 
Step 4: The model produced output until the combined token count reached the 4096‑token limit. At token 4096 the generation was automatically truncated and the response was terminated. 
Step 5: The guardrail correctly triggered a safe stop and logged an event indicating that the token budget was reached. No error or policy‑violating content was produced. 
Step 6: Findings documented: the budget guardrail functions as expected, halting generation precis

## 13. Implementation-plan checklist

| Requirement | This notebook |
|---|---|
| Groq / Llama alternative to local Ollama | ✅ Groq + Llama 3.x |
| Planner → Executor → Reviewer | ✅ |
| LangGraph | ✅ |
| CrewAI | ✅ |
| `execution_count` in graph state | ✅ |
| Conditional edge | ✅ |
| Stop when `execution_count > 3` | ✅ |
| Safe fallback | ✅ |
| PydanticAI-based strict schemas | ✅ PydanticAI package + strict Pydantic schemas |
| Mem0 OSS/local | ✅ |
| Local SQLite persistence | ✅ `mem0_history.db` |
| Persistent user state across sessions | ✅ same `user_id` + local DB |
| Colab compatible | ✅ Groq API + native async execution |